# Classes and objects (OOP part 1)

**Goal:** bundle **data** and the **functions that work on that data** into one reusable thing: a class.

We continue with the webshop data from yesterday (`sales_raw.csv`, `products.json`).

## 1. Why classes? The problem with dicts

A dict holds data, but the functions that use it live somewhere else and nothing stops a typo.

In [ ]:
product = {"name": "Laptop", "category": "Electronics", "unit_price": 9990}

def price_with_vat(product, rate=0.25):
    return product["unit_price"] * (1 + rate)

print(price_with_vat(product))

In [ ]:
# This cell raises an error on purpose
bad_product = {"name": "Mouse", "categroy": "Electronics", "unit_price": 299}   # typo in the key
bad_product["category"]

## 2. Our first class

- a **class** is a blueprint
- an **object** (or *instance*) is one thing built from the blueprint
- `__init__` is the **setup method**: it runs automatically when an object is created
- `self` is the object itself; things stored on `self` are **attributes**

*(`__init__` has two underscores on each side. It is the only special method we use in this course.)*

In [ ]:
class Product:
    def __init__(self, name, category, unit_price):
        self.name = name
        self.category = category
        self.unit_price = unit_price

laptop = Product("Laptop", "Electronics", 9990)
mouse = Product("Mouse", "Electronics", 299)

print(laptop.name, laptop.unit_price)
print(mouse.name, mouse.unit_price)

Each object has its **own** attributes:

In [ ]:
mouse.unit_price = 249     # change one object...
print(mouse.unit_price)
print(laptop.unit_price)   # ...the other is unaffected

## 3. Methods: functions that belong to the class

A method is a function defined inside a class. Its first parameter is always `self`.

In [ ]:
class Product:
    def __init__(self, name, category, unit_price):
        self.name = name
        self.category = category
        self.unit_price = unit_price

    def price_with_vat(self, rate=0.25):
        return self.unit_price * (1 + rate)

    def apply_discount(self, percent):
        self.unit_price = self.unit_price * (1 - percent / 100)

laptop = Product("Laptop", "Electronics", 9990)
print(laptop.price_with_vat())
laptop.apply_discount(10)
print(laptop.unit_price)

Terminology (see the handout): when we **define** `price_with_vat(self, rate=0.25)`, `rate` is a *parameter*. When we **call** `laptop.price_with_vat(0.12)`, `0.12` is the *argument*. `self` is filled in automatically with the object before the dot.

## 4. A `describe()` method

It is handy if an object can describe itself in a readable way. We simply write a method that returns a text.

In [ ]:
class Product:
    def __init__(self, name, category, unit_price):
        self.name = name
        self.category = category
        self.unit_price = unit_price

    def price_with_vat(self, rate=0.25):
        return self.unit_price * (1 + rate)

    def describe(self):
        return f"{self.name} ({self.category}): {self.unit_price:.2f} SEK"

laptop = Product("Laptop", "Electronics", 9990)
print(laptop.describe())

## 5. Objects in lists

Because objects have behaviour, loops get readable.

In [ ]:
import json

with open("products.json", encoding="utf-8") as f:
    raw = json.load(f)

products = []
for p in raw:
    products.append(Product(p["name"], p["category"], p["unit_price"]))

for p in products:
    print(p.describe())

Sorting objects: tell `sorted()` *what* to sort by with a small helper function.

In [ ]:
def get_price(product):
    return product.unit_price

most_expensive = sorted(products, key=get_price, reverse=True)
for p in most_expensive[:3]:
    print(p.describe())

## 6. Objects that hold other objects

An order contains several products with quantities. Each line of the order is a small dict (week 1!).

In [ ]:
class Order:
    def __init__(self, order_id, customer):
        self.order_id = order_id
        self.customer = customer
        self.lines = []                      # list of dicts: {"product": ..., "quantity": ...}

    def add_item(self, product, quantity=1):
        self.lines.append({"product": product, "quantity": quantity})

    def total(self):
        total = 0
        for line in self.lines:
            total = total + line["product"].unit_price * line["quantity"]
        return total

    def describe(self):
        return f"Order {self.order_id} for {self.customer}: {len(self.lines)} lines, {self.total():.2f} SEK"

products_by_name = {}
for p in products:
    products_by_name[p.name] = p

order = Order(2001, "Anna")
order.add_item(products_by_name["Laptop"])
order.add_item(products_by_name["Mouse"], 2)
print(order.describe())

## 7. You have been using objects all along

Everything in Python is an object: it has a **type** (its class), **attributes** and **methods**.

In [ ]:
text = "hello"
print(type(text))
print(text.upper())          # a method of the str class
numbers = [3, 1, 2]
numbers.append(0)            # a method of the list class
print(numbers)

Later, a pandas `DataFrame` is an object too: `df.shape` is an attribute, `df.describe()` a method. Now you know how they are built.

## 8. Class attributes: shared by all objects

In [ ]:
class Product:
    VAT_RATE = 0.25                       # belongs to the class, shared by all products

    def __init__(self, name, category, unit_price):
        self.name = name
        self.category = category
        self.unit_price = unit_price

    def price_with_vat(self):
        return self.unit_price * (1 + self.VAT_RATE)

print(Product("Pen", "Stationery", 20).price_with_vat())

## 9. Common mistakes

1. Forgetting `self` as the first parameter of a method
2. Writing `unit_price = unit_price` instead of `self.unit_price = unit_price` in `__init__`
3. Forgetting the brackets when calling a method: `order.total` is the method itself, `order.total()` calls it

In [ ]:
# This cell raises an error on purpose
class Broken:
    def say_hi():          # forgot self
        return "hi"

Broken().say_hi()

## Mini exercise

Create a class `Customer` with `name` and a list `orders`. Add `add_order(order)` and `total_spent()` (use a loop). Create a customer and add the order from section 6.

In [ ]:
# your code here